In [ ]:
# Extract_missing_pin

> Extract missing pin from a list of pins

In [ ]:
#| default_exp extract_missing_pin

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cv2
from typing import NewType, List, Tuple, Union, Dict
from pathlib import Path
from scipy.ndimage import label
from tqdm.auto import tqdm
from argparse import ArgumentParser, BooleanOptionalAction

In [ ]:
OpenCvImage = NewType('OpenCvImage', np.ndarray)

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| export
from private_easy_pin_detection.core import *
from private_easy_pin_detection.pin_map import *

In [ ]:
#| export    
def get_full_image_smaller_mask(
        img:OpenCvImage,# actual image
        msk:OpenCvImage, # mask where pins are small or no pins are found
        im_name:str,
        recipe_no:str,
        col_bbox:List[Tuple[int, int, int, int]],
        corrected_msk:OpenCvImage=None, # In case if from other model or any how if corrected masks is also available
        save_im_path:Union[str, None]=None,
        save_msk_path:Union[str, None]=None,
        save_corrected_msk_path:Union[str, None]=None,
        minimum_area:int=9000
   )->Tuple[OpenCvImage, OpenCvImage]:
  """
  Get image of all the columns and saving pins where the mask is smaller than specific number
  """
  # getting bounding boxes for each column

  if save_im_path is not None:
    Path(save_im_path).mkdir(exist_ok=True, parents=True)

  if save_msk_path is not None:
    Path(save_msk_path).mkdir(exist_ok=True, parents=True)

  if save_corrected_msk_path is not None:
    Path(save_corrected_msk_path).mkdir(exist_ok=True, parents=True)

  num_cols_ = split_image_no()[recipe_no]
  pin_map_func =f'pin_map{recipe_no}'

  last_mask = np.zeros_like(img)
  pin_index_func=globals()[pin_map_func]
  all_cols = []
  num_cols = len(pin_index_func().keys())

  for col_no in range(num_cols):
        col_bbox_coord = col_bbox[col_no]

        col_data = img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        col_data_mask = msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        if corrected_msk is not None:
          col_data_corrected_mask = corrected_msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]

        pin_indexes = pin_index_func()[col_no]
        # getting image and mask part
        single_col_parts = split_image(
                            col_data, 
                            num_cols_, 
                            'vertical')

        single_col_parts_msk = split_image(
                      col_data_mask, 
                      num_cols_, 
                      'vertical')
        if corrected_msk is not None:
          single_col_parts_corrected_msk = split_image(col_data_corrected_mask, num_cols_, 'vertical')

        parts=[]
        for idx, part in enumerate(single_col_parts):
            if idx in pin_indexes:
                part = part
                msk_part = single_col_parts_msk[idx]
                _, n_f = label(msk_part)
                if (save_im_path is not None and cv2.countNonZero(msk_part)<minimum_area)\
                  or (save_im_path is not None and n_f>1):
                    cv2.imwrite(
                       f'{save_im_path}/{Path(im_name).stem}_col_{col_no}_{idx}.png', 
                       part)
                    if save_msk_path is not None:
                        cv2.imwrite(
                           f'{save_msk_path}/{Path(im_name).stem}_col_{col_no}_{idx}.png', 
                           msk_part)
                    if save_corrected_msk_path is not None:
                        cv2.imwrite(
                           f'{save_corrected_msk_path}/{Path(im_name).stem}_col_{col_no}_{idx}.png',
                             single_col_parts_corrected_msk[idx])
            else:
                #TODO: add pin mask
                part = np.zeros_like(part)
            parts.append(part)
        sn_col=np.concatenate(parts, axis=0)

        last_mask[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = sn_col
        all_cols.append(sn_col)
  full_part = np.concatenate(all_cols, axis=1)
  return last_mask, full_part


In [ ]:
#| export
def extract_smaller_mask_pin(
        im_path:str,
        msk_path:str,
        corrected_msk_path:str,
        tmp_path:str,
        save_im_path:Union[str, Path]=None,
        save_msk_path:Union[str, Path]=None,
        save_corrected_msk_path:Union[str, Path]=None,
        min_msk_area:int=300,      
    ):
    'Check mask of each pin, if it is smaller than specific area, save it to the specific folder'
    
    # getting all  images from path
    img = read_img(im_path)
    msk = read_img(msk_path)

    if corrected_msk_path is not None:
        cor_msk = read_img(corrected_msk_path)
    cor_msk = None
    recipe_no = get_m_name(f'{im_path}')

    tmp_img = read_img(f'{tmp_path}/{recipe_no}.png')
    
    # getting bouding box based on recipe 
    bbox_func = globals()[f'get_all_columns_{recipe_no}']
    col_bbox = bbox_func(
        tst_img=img,
        tmp_img=tmp_img
    )
    # Now saving all the parts which has smaller mask


    zoomed_, full_ = get_full_image_smaller_mask(
        img=img,
        msk=msk,
        corrected_msk=cor_msk,
        im_name=Path(im_path).name,
        recipe_no=recipe_no,
        col_bbox=col_bbox,
        save_im_path=save_im_path,
        save_msk_path=save_msk_path,
        save_corrected_msk_path=save_corrected_msk_path,
        minimum_area=min_msk_area
                                )

In [ ]:
im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/im2_test_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/missing/images/VFV4.7.9.5_2024030211055846_ID_00187043797818157132408_In_17_r_1_FRONT_Missing Lead_image2.png')
msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/im2_test_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/missing/masks/VFV4.7.9.5_2024030211055846_ID_00187043797818157132408_In_17_r_1_FRONT_Missing Lead_image2.png')
save_im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/im2_test_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/missing/missing_pin_sn_images')
tmp_path = os.environ['TEMP_PATH']
extract_smaller_mask_pin(
                        im_path=im_path,
                        msk_path=msk_path,
                        corrected_msk_path=None,
                        tmp_path=tmp_path,
                        save_im_path=save_im_path,
                        save_msk_path=None,
                        save_corrected_msk_path=None,
                        min_msk_area=300
)

In [ ]:
#| export
def extract_smaller_mask_pin_folder(
    im_path:Union[str, Path],
    msk_path:Union[str, Path],
    cor_msk_path:Union[str, Path], # corrected mask path
    save_im_path:Union[str, Path]=None,
    save_msk_path:Union[str, Path]=None,
    save_cor_msk_path:Union[str, Path]=None,
    tmp_path:Union[str, Path]=None,
    min_msk_area:int=330
   ):
    """
    Extract path from folder
    """
    images = Path(im_path).ls()
    im_names = get_name_(images)




    print(f' NUmber of images found = {len(im_names)}')
    for i in tqdm(im_names, total=len(im_names)):
        print(i)
        im_ = Path(im_path, i)
        recipe_name = get_m_name(i)
        tmp_path = tmp_path

        msk_ = Path(msk_path, i)
        if cor_msk_path is not None:
            cor_msk_path = Path(cor_msk_path, i)
        else:
            cor_msk_path = None
        extract_smaller_mask_pin(
            im_path=im_,
            msk_path=msk_,
            corrected_msk_path=cor_msk_path,
            tmp_path=tmp_path,
            min_msk_area=min_msk_area,
            save_msk_path=save_msk_path,
            save_corrected_msk_path=save_cor_msk_path,
            save_im_path=save_im_path
        )

In [ ]:
im_path=Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/im2_test_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/missing/images')
msk_path=Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/im2_test_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/missing/masks')
cor_msk_path=None
save_im_path=Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/im2_test_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/missing/missing_pin_sn_images')
save_msk_path=None
save_cor_msk_path=None
min_msk_area=300


In [ ]:
extract_smaller_mask_pin_folder(
    im_path=im_path,
    msk_path=msk_path,
    cor_msk_path=cor_msk_path,
    save_im_path=save_im_path,
    save_msk_path=save_msk_path,
    save_cor_msk_path=save_cor_msk_path,
    min_msk_area=min_msk_area,
    tmp_path=os.environ['TEMP_PATH']
)

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument(
        '--im_path', 
        type=str, 
        help='Path to image'
        )
    parser.add_argument(
        '--msk_path', 
        type=str, 
        help='Path to mask')

    parser.add_argument(
        '--cor_msk_path', 
        type=str, 
        help='Path to mask')
    parser.add_argument(
        '--tmp_path', 
        type=str, 
        help='Path to template path so without filename,\
              inside this folder there should be an image called recipe_no[e.g.87.png]')
    parser.add_argument(
        '--save_im_path', 
        type=str, 
        help='Path to save the images')
    parser.add_argument(
        '--save_cor_msk_path', 
        type=str, 
        help='Path to save the images')
    parser.add_argument(
        '--min_area', 
        type=int, 
        default=1, 
        help='Minimum area of mask, if the area is less than that, then  mask to be saved'
        )
    return parser.parse_args()

In [ ]:
#| export
def main():
    args = parse_args_()
    extract_smaller_mask_pin_folder(
        im_path=args.im_path,
        msk_path=args.msk_path,
        cor_msk_path=args.cor_msk_path,
        tmp_path=args.tmp_path,
        save_im_path=args.save_im_path,
        save_cor_msk_path=args.save_cor_msk_path,
        min_msk_area=args.min_area
    )

In [ ]:
#| export
if __name__ == '__main__':
    main()

In [ ]:
#| hide
import nbdev;nbdev.nbdev_export('20_extract_missing_pins.ipynb')